In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/


# AzureSQL Connector Samples

Read/write samples for Azure SQL Database using the `AZURE_SQLSERVER` connector type. Replace all placeholders before running a sample.


## Ingestion Read


In [ ]:
azuresql_df = spark.read.format("aidataplatform") \
    .option("type", "AZURE_SQLSERVER") \
    .option("host", "<HOST>") \
    .option("port", "1433") \
    .option("database.name", "<DATABASE_NAME>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TABLE_NAME>") \
    .load()

azuresql_df.show()


## Ingestion Write

Use `CREATE`, `APPEND`, `OVERWRITE`, or `MERGE` as appropriate. `write.merge.keys` is required for `MERGE`.


In [ ]:
azuresql_df.write.format("aidataplatform") \
    .option("type", "AZURE_SQLSERVER") \
    .option("host", "<HOST>") \
    .option("port", "1433") \
    .option("database.name", "<DATABASE_NAME>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TARGET_TABLE_NAME>") \
    .option("write.mode", "CREATE") \
    .save()


## External Catalog and `catalog.id`

Create an AzureSQL external catalog in **Master Catalogs** first. Use three-part names for catalog access, or `catalog.id` to reuse its connection in ingestion-style operations.


In [ ]:
# External catalog read and write
catalog_azuresql_df = spark.table("<CATALOG_NAME>.<SCHEMA>.<TABLE_NAME>")
catalog_azuresql_df.show()
azuresql_df.write.saveAsTable("<CATALOG_NAME>.<SCHEMA>.<TARGET_TABLE_NAME>")

# Ingestion-style read and write through catalog.id
catalog_id_df = spark.read.format("aidataplatform") \
    .option("catalog.id", "<CATALOG_ID>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TABLE_NAME>") \
    .load()

catalog_id_df.write.format("aidataplatform") \
    .option("catalog.id", "<CATALOG_ID>") \
    .option("schema", "<SCHEMA>") \
    .option("table", "<TARGET_TABLE_NAME>") \
    .option("write.mode", "APPEND") \
    .save()


## Pushdown SQL


In [ ]:
azuresql_pushdown_df = spark.read.format("aidataplatform") \
    .option("type", "AZURE_SQLSERVER") \
    .option("host", "<HOST>") \
    .option("port", "1433") \
    .option("database.name", "<DATABASE_NAME>") \
    .option("user.name", "<USERNAME>") \
    .option("password", "<PASSWORD>") \
    .option("pushdown.sql", "SELECT TOP 10 * FROM <SCHEMA>.<TABLE_NAME>") \
    .load()

azuresql_pushdown_df.show()


## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `type` | `AZURE_SQLSERVER` | Y for inline configuration | Azure SQL Database connector type. |
| `catalog.id` | Existing external catalog identifier | N | Reuses connection details from an external catalog. |
| `host`, `port`, `database.name` | Valid Azure SQL endpoint values | Y for inline configuration | Server endpoint and database name. |
| `user.name`, `password` | Valid SQL credentials | Y for inline configuration | Credentials used for connectivity. |
| `schema`, `table`, `view`, `sql` | Valid source selection | `schema` and `table` are Y for table access | Select a table, view, or query. |
| `fetch.size`, `row.limit` | Positive integer | N | Read batching and limiting. |
| `partition.column`, `partition.num`, `partition.lower`, `partition.upper` | Partition settings | N | Range-partitioned reads or writes. |
| `pushdown.columns`, `pushdown.filter`, `pushdown.sql` | Valid projection, predicate, or SQL | N | Push work down to Azure SQL. |
| `write.mode` | `CREATE`, `APPEND`, `OVERWRITE`, `MERGE` | Y during writes | Write behavior. |
| `write.merge.keys` | Comma-separated columns | Y for `MERGE` | Match keys for merge writes. |
| `write.empty.value.as.null`, `write.batch.size`, `write.reject.limit` | Connector-supported values | N | Write controls. |
| `login.timeout.seconds` | Integer from `1` to `300` | N | JDBC login timeout. |
